# TF expression versus activity: significant-TF counts

This notebook reads saved TF-level results from `08compare_TF_expression_vs_activity_counts_use_sameTF.R`. It creates separate directional barplots for the linear-model and Mann–Whitney tests. Each plot shows the number of higher-in-late-AD and lower-in-late-AD TFs per celltype–region, with Expression and Activity bars side by side.

A count table is exported for each test and for the union of the two tests. If a TF has significant results in opposite directions across the two tests, it contributes to both directional union counts, once to the total, and is flagged as a direction conflict. The overall paired comparisons retain Wilcoxon annotations and lines connecting matched contexts; they have no boxplots. Plot-ready tables are saved under `figure_output_dir/plot_data`.


In [ ]:
suppressPackageStartupMessages({
  library(ggplot2)
  library(dplyr)
  library(tidyr)
  library(readr)
  library(egg)
})

analysis_output_dir <- "/project/zduren/durenlab/naqing_palmetto/REGA_separate/07_tf/results/expression_activity_counts_same_tf"
long_results_file <- file.path(analysis_output_dir, "TF_expression_activity_tests_long.tsv")
summary_file <- file.path(analysis_output_dir, "significant_TF_count_summary_all.tsv")
paired_tests_file <- file.path(analysis_output_dir, "overall_expression_vs_activity_statistical_tests.tsv")

figure_output_dir <- file.path(analysis_output_dir, "figures_only_notebook")
plot_data_dir <- file.path(figure_output_dir, "plot_data")
dir.create(plot_data_dir, recursive = TRUE, showWarnings = FALSE)

significance_column <- "p_adj"
alpha <- 0.05
save_figures <- TRUE
png_dpi <- 300
font_family <- "sans"
expr_color <- "#E76F51"
act_color <- "#159D9C"

directional_width <- 14
# Height grows for one row per celltype-region in each direction facet.
directional_height <- 12
linear_paired_width <- 4.3
linear_paired_height <- 4.3
linear_paired_base_size <- 13
mw_paired_width <- 4.3
mw_paired_height <- 4.3
mw_paired_base_size <- 13

for (f in c(long_results_file, summary_file, paired_tests_file)) {
  if (!file.exists(f)) stop("Required source output is missing; run the analysis script first: ", f)
}
cat("Reading TF-level results from:\n", long_results_file, "\n")
cat("Saving figures and plot tables under:\n", figure_output_dir, "\n")


In [ ]:
tf_results <- read.delim(long_results_file, check.names = FALSE, stringsAsFactors = FALSE)
summary_df <- read.delim(summary_file, check.names = FALSE, stringsAsFactors = FALSE)
paired_test_summary <- read.delim(paired_tests_file, check.names = FALSE, stringsAsFactors = FALSE)

required_long_columns <- c("TF", "celltype", "region", "celltype_region", "data_type", "method", "effect", significance_column)
missing_columns <- setdiff(required_long_columns, names(tf_results))
if (length(missing_columns) > 0) stop("TF-level results missing columns: ", paste(missing_columns, collapse = ", "))

# Significant TFs are defined by the same BH-FDR threshold as the source script.
# Positive effect means higher late-AD values; negative effect means lower.
significant_details <- tf_results %>%
  filter(method %in% c("Linear model", "Mann-Whitney")) %>%
  filter(is.finite(effect), effect != 0, !is.na(.data[[significance_column]]), .data[[significance_column]] < alpha) %>%
  mutate(direction = if_else(effect > 0, "Higher in late AD", "Lower in late AD")) %>%
  distinct(method, celltype, region, celltype_region, data_type, TF, direction, .keep_all = TRUE)

method_counts <- significant_details %>%
  group_by(test = method, celltype, region, celltype_region, data_type) %>%
  summarise(
    n_up = n_distinct(TF[direction == "Higher in late AD"]),
    n_down = n_distinct(TF[direction == "Lower in late AD"]),
    n_total = n_distinct(TF),
    n_direction_conflict = 0L,
    .groups = "drop"
  )

# A TF is in the union if it is significant in either test. For union counts,
# up/down are directional set unions; a discordant TF may count in both.
union_tf <- significant_details %>%
  group_by(celltype, region, celltype_region, data_type, TF) %>%
  summarise(
    is_up = any(direction == "Higher in late AD"),
    is_down = any(direction == "Lower in late AD"),
    .groups = "drop"
  )

union_counts <- union_tf %>%
  group_by(celltype, region, celltype_region, data_type) %>%
  summarise(
    n_up = sum(is_up),
    n_down = sum(is_down),
    n_total = n(),
    n_direction_conflict = sum(is_up & is_down),
    .groups = "drop"
  ) %>%
  mutate(test = "Union of tests")

context_keys <- tf_results %>% distinct(region, celltype, celltype_region) %>% arrange(region, celltype)
context_order <- context_keys$celltype_region
all_tests <- c("Linear model", "Mann-Whitney", "Union of tests")
count_grid <- merge(
  context_keys,
  expand.grid(data_type = c("Expression", "Activity"), test = all_tests, stringsAsFactors = FALSE),
  by = NULL
)

directional_counts_by_context <- bind_rows(method_counts, union_counts) %>%
  right_join(count_grid, by = c("celltype", "region", "celltype_region", "data_type", "test")) %>%
  mutate(
    n_up = coalesce(n_up, 0L),
    n_down = coalesce(n_down, 0L),
    n_total = coalesce(n_total, 0L),
    n_direction_conflict = coalesce(n_direction_conflict, 0L),
    celltype_region = factor(celltype_region, levels = rev(context_order)),
    data_type = factor(data_type, levels = c("Expression", "Activity")),
    test = factor(test, levels = all_tests)
  ) %>%
  arrange(test, data_type, celltype_region)

directional_counts_overall <- directional_counts_by_context %>%
  group_by(test, data_type) %>%
  summarise(
    n_up_TF_contexts = sum(n_up),
    n_down_TF_contexts = sum(n_down),
    n_total_TF_contexts = sum(n_total),
    n_direction_conflict_TF_contexts = sum(n_direction_conflict),
    n_celltype_regions = n_distinct(celltype_region),
    .groups = "drop"
  )

direction_conflicts <- union_tf %>%
  filter(is_up & is_down) %>%
  arrange(celltype_region, data_type, TF)

write_tsv(significant_details, file.path(plot_data_dir, "directional_significant_TF_details_by_test.tsv"))
write_tsv(union_tf, file.path(plot_data_dir, "directional_significant_TF_union_details.tsv"))
write_tsv(direction_conflicts, file.path(plot_data_dir, "direction_conflicting_significant_TFs.tsv"))
write_tsv(directional_counts_by_context, file.path(plot_data_dir, "directional_significant_TF_counts_by_context.tsv"))
write_tsv(directional_counts_overall, file.path(plot_data_dir, "directional_significant_TF_counts_overall.tsv"))

# For paired overall plots use each test's saved significance counts, keeping
# contexts with at least one significant TF and both matched data types.
paired_plot_df <- summary_df %>%
  group_by(method, celltype_region) %>%
  filter(sum(n_significant, na.rm = TRUE) > 0) %>%
  filter(n() == 2, n_distinct(data_type) == 2) %>%
  ungroup() %>%
  mutate(data_type = factor(data_type, levels = c("Expression", "Activity")))
write_tsv(paired_plot_df, file.path(plot_data_dir, "paired_overall_significant_TF_counts.tsv"))
write_tsv(paired_test_summary, file.path(plot_data_dir, "paired_wilcoxon_test_annotations.tsv"))

cat("Context count:", length(context_order), "\n")
cat("Directional count rows:", nrow(directional_counts_by_context), "\n")
cat("Direction-conflict rows in the union:", nrow(direction_conflicts), "\n")
print(directional_counts_overall, n = Inf)


In [ ]:
save_plot <- function(plot_obj, stem, width, height) {
  if (!save_figures) return(invisible(NULL))
  ggsave(file.path(figure_output_dir, paste0(stem, ".pdf")), plot_obj, width = width, height = height, units = "in", limitsize = FALSE)
  ggsave(file.path(figure_output_dir, paste0(stem, ".png")), plot_obj, width = width, height = height, units = "in", dpi = png_dpi, limitsize = FALSE)
}

plot_directional_counts <- function(test_name) {
  dat <- directional_counts_by_context %>%
    filter(test == test_name) %>%
    pivot_longer(c(n_up, n_down), names_to = "direction_key", values_to = "n_significant_TF") %>%
    mutate(direction = recode(direction_key, n_up = "Higher in late AD", n_down = "Lower in late AD"))

  ggplot(dat, aes(x = n_significant_TF, y = celltype_region, fill = data_type)) +
    geom_col(position = position_dodge(width = 0.78), width = 0.68, color = "black", linewidth = 0.2) +
    facet_grid(. ~ direction) +
    scale_fill_manual(values = c("Expression" = expr_color, "Activity" = act_color)) +
    scale_x_continuous(breaks = scales::pretty_breaks(n = 6), expand = expansion(mult = c(0, 0.05))) +
    labs(x = "Number of significant TFs", y = NULL, fill = NULL, title = test_name) +
    theme_article(base_size = 15) +
    theme(
      text = element_text(family = font_family),
      legend.position = "top",
      strip.text = element_text(face = "bold", size = 14),
      panel.grid.major.y = element_blank(),
      panel.grid.minor = element_blank(),
      axis.text.y = element_text(size =13),
      plot.title = element_text(face = "bold"),
      plot.margin = margin(10, 12, 10, 10)
    )
}

for (test_name in c("Linear model", "Mann-Whitney")) {
  test_stub <- if (test_name == "Linear model") "linear_model" else "mann_whitney"
  test_plot_data <- directional_counts_by_context %>% filter(test == test_name)
  write_tsv(test_plot_data, file.path(plot_data_dir, paste0(test_stub, "_directional_barplot_data.tsv")))

  p_directional <- plot_directional_counts(test_name)
  options(repr.plot.width = directional_width, repr.plot.height = directional_height, repr.plot.res = 120)
  print(p_directional)
  save_plot(p_directional, paste0("directional_significant_TF_counts_", test_stub), directional_width, directional_height)
}


In [ ]:
format_p <- function(p) {
  if (length(p) == 0 || is.na(p) || !is.finite(p)) return("NA")
  if (p < 0.001) return(formatC(p, format = "e", digits = 2))
  formatC(p, format = "f", digits = 3)
}

plot_paired_counts <- function(method_name, base_size = 13) {
  dat <- filter(paired_plot_df, method == method_name)
  if (nrow(dat) == 0) stop("No paired count data for method: ", method_name)
  test_row <- paired_test_summary %>% filter(method == method_name)
  p_value <- if (nrow(test_row) == 1) test_row$paired_wilcoxon_p[1] else NA_real_
  p_label <- paste0("Paired Wilcoxon p = ", format_p(p_value))

  # Keep the annotation bracket below the label with reserved vertical space.
  y_max <- max(dat$n_significant, na.rm = TRUE)
  y_span <- max(1, y_max)
  bracket_y <- y_max + 0.16 * y_span
  label_y <- y_max + 0.38 * y_span
  lower_tick <- bracket_y - 0.06 * y_span
  upper_limit <- y_max + 0.62 * y_span

  ggplot(dat, aes(x = data_type, y = n_significant, group = celltype_region)) +
    geom_line(color = "grey65", linewidth = 0.5, alpha = 0.7) +
    geom_boxplot(
      aes(group = data_type, color = data_type),
      width = 0.55,
      outlier.shape = NA,
      fill = NA,
      linewidth = 0.6
    ) +
    geom_point(aes(color = data_type), shape = 16, size = 2.5, alpha = 0.9, position = position_jitter(width = 0.035, height = 0)) +
    scale_color_manual(values = c("Expression" = expr_color, "Activity" = act_color), guide = "none") +
    scale_y_continuous(breaks = scales::pretty_breaks(n = 6), expand = expansion(mult = c(0, 0))) +
    coord_cartesian(ylim = c(0, upper_limit), clip = "off") +
    labs(x = NULL, y = "Number of significant TFs", title = method_name, subtitle = paste0("Matched celltype-region contexts; n = ", n_distinct(dat$celltype_region), "; lines connect paired counts")) +
    theme_article(base_size = base_size) +
    theme(
      text = element_text(family = font_family),
      legend.position = "none",
      panel.grid.major.x = element_blank(),
      panel.grid.minor = element_blank(),
      plot.title = element_text(face = "bold"),
      plot.margin = margin(10, 12, 10, 10)
    ) +
    annotate("segment", x = 1, xend = 2, y = bracket_y, yend = bracket_y, linewidth = 0.45) +
    annotate("segment", x = 1, xend = 1, y = lower_tick, yend = bracket_y, linewidth = 0.45) +
    annotate("segment", x = 2, xend = 2, y = lower_tick, yend = bracket_y, linewidth = 0.45) +
    annotate("text", x = 1.5, y = label_y, label = p_label, size = 3.5, family = font_family)
}

# Separate calls and display settings make each paired figure easy to tune.
linear_paired_data <- filter(paired_plot_df, method == "Linear model")
write_tsv(linear_paired_data, file.path(plot_data_dir, "linear_model_paired_plot_data.tsv"))
p_linear_paired <- plot_paired_counts("Linear model", base_size = linear_paired_base_size)
options(repr.plot.width = linear_paired_width, repr.plot.height = linear_paired_height, repr.plot.res = 120)
print(p_linear_paired)
save_plot(p_linear_paired, "overall_paired_counts_linear_model", linear_paired_width, linear_paired_height)

mw_paired_data <- filter(paired_plot_df, method == "Mann-Whitney")
write_tsv(mw_paired_data, file.path(plot_data_dir, "mann_whitney_paired_plot_data.tsv"))
p_mw_paired <- plot_paired_counts("Mann-Whitney", base_size = mw_paired_base_size)
options(repr.plot.width = mw_paired_width, repr.plot.height = mw_paired_height, repr.plot.res = 120)
print(p_mw_paired)
save_plot(p_mw_paired, "overall_paired_counts_mann_whitney", mw_paired_width, mw_paired_height)

cat("Finished. All figure input tables are saved under:\n", plot_data_dir, "\n")


In [ ]:

directional_height <- 12
linear_paired_width <- 3.5
linear_paired_height <- 4.3
linear_paired_base_size <- 13
mw_paired_width <- 3.5
mw_paired_height <- 4.3
mw_paired_base_size <- 13
